# bigcompute.science Research Agent + GPU Compute

**You have a free T4 GPU right now.** This notebook lets you:
1. **Compile & run** CUDA experiments on open math conjectures (Zaremba, Kronecker, Ramanujan, class numbers)
2. **Review findings** with AI peer review (Gemini free, or OpenAI/Anthropic)
3. **Contribute results** back via PR

This is distributed computational mathematics research. Every Colab instance that runs an experiment extends the frontier.

**API key:** Get a free Gemini key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (instant, no credit card). Or use OpenAI/Anthropic.

> Part of [bigcompute.science](https://bigcompute.science) — open computational mathematics.

In [ ]:
# === Step 1: Clone + install ===
!git clone https://github.com/cahlen/idontknow.git 2>/dev/null || (cd idontknow && git pull)
%cd idontknow
!pip install -q httpx

In [ ]:
# === Step 2: Set your API key ===
# The agent tries providers in order: Anthropic -> OpenAI -> Gemini
# In Colab, Gemini is the easiest (free, you're already logged into Google)

import os

# --- Option A: Colab Secrets (recommended) ---
# Click the key icon in the sidebar, add GEMINI_API_KEY
try:
    from google.colab import userdata
    for key_name in ['GEMINI_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
        try:
            val = userdata.get(key_name)
            if val:
                os.environ[key_name] = val
                print(f'  Loaded {key_name} from Colab Secrets')
        except Exception:
            pass  # Secret not set, that's fine
except ImportError:
    pass  # Not in Colab

# --- Option B: Paste directly ---
# Get a free Gemini key: https://aistudio.google.com/apikey
# os.environ['GEMINI_API_KEY'] = 'AIza...'       # Gemini (free)
# os.environ['OPENAI_API_KEY'] = 'sk-proj-...'   # OpenAI
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...' # Anthropic

has = {k: bool(os.environ.get(k)) for k in ['GEMINI_API_KEY','GOOGLE_API_KEY','OPENAI_API_KEY','ANTHROPIC_API_KEY']}
available = [k.replace('_API_KEY','') for k,v in has.items() if v]
print(f'\nAvailable: {", ".join(available) if available else "NONE"}')
if not available:
    print('\nGet a free Gemini key: https://aistudio.google.com/apikey')
    print('Then paste above or add as Colab Secret')


In [ ]:
# === Step 3: Quick test — can the agent talk to an LLM? ===
import sys
sys.path.insert(0, '.')
from scripts.research_agent import call_llm

result = call_llm('Respond with exactly: {"status": "ok", "model": "your-name"}', 'test')
if result:
    print(f'LLM connection: OK')
    print(f'Response: {result[:100]}')
else:
    print('FAILED — check your API key above')

In [ ]:
# === Step 4: Dry run — see what the agent would do ===
!python3 scripts/research_agent.py --once --dry-run

In [ ]:
# === Step 5: Harvest results (see what experiments completed) ===
!python3 scripts/research_agent.py --once --phase harvest

In [ ]:
# === Step 6: Compile CUDA kernels (auto-detects your GPU) ===
import subprocess

# Auto-detect GPU architecture
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                           capture_output=True, text=True)
    gpu_name, compute_cap = result.stdout.strip().split(', ')
    major, minor = compute_cap.split('.')
    sm = f'sm_{major}{minor}'
    print(f'Detected: {gpu_name} (compute {compute_cap} → {sm})')
except:
    sm = 'sm_75'  # fallback to T4
    print(f'GPU detection failed, using {sm} (T4 default)')

# Known GPU architectures:
# T4 = sm_75, A100 = sm_80, L4 = sm_89, RTX 4090 = sm_89
# H100 = sm_90, B200 = sm_100a, RTX 5090 = sm_120a

!nvidia-smi | head -5
print(f'\nCompiling with -arch={sm}...')

import os
kernels = [
    ('zaremba_density_gpu', 'scripts/experiments/zaremba-density/zaremba_density_gpu.cu', '-lm'),
    ('kronecker_gpu', 'scripts/experiments/kronecker-coefficients/kronecker_gpu.cu', '-lm'),
    ('ramanujan_gpu', 'scripts/experiments/ramanujan-machine/ramanujan_gpu.cu', '-lm'),
    ('class_v2', 'scripts/experiments/class-numbers/class_numbers_v2.cu', '-lpthread -lm'),
]

compiled = []
for binary, source, libs in kernels:
    if os.path.exists(source):
        ret = os.system(f'nvcc -O3 -arch={sm} -o {binary} {source} {libs} 2>&1')
        status = 'OK' if ret == 0 else 'FAILED'
        print(f'  {binary}: {status}')
        if ret == 0:
            compiled.append(binary)
    else:
        print(f'  {binary}: source not found')

print(f'\n{len(compiled)}/{len(kernels)} kernels compiled for {sm}.')


In [ ]:
# === Step 11: Run an experiment! ===
# Pick one and run it. Results will be saved to a log file.
# Each extends the frontier of verified mathematical knowledge.

import os
os.makedirs('scripts/experiments/zaremba-density/results', exist_ok=True)

# --- Option A: Zaremba density (quick, ~2-30 min depending on range) ---
# Try a digit set that hasn't been explored at this range yet
experiment = 'zaremba'  # @param ['zaremba', 'kronecker', 'ramanujan', 'class_numbers']

if experiment == 'zaremba':
    # Extend the {1,2,k} hierarchy — try k=8 at 10^10
    !./zaremba_density_gpu 10000000000 1,2,8 | tee scripts/experiments/zaremba-density/results/gpu_A128_1e10_colab.log

elif experiment == 'kronecker':
    # Compute S_20 Kronecker coefficients (quick, ~4 sec)
    !python3 scripts/experiments/kronecker-coefficients/char_table.py 20
    !./kronecker_gpu 20 | tee scripts/experiments/kronecker-coefficients/results/kronecker_n20_colab.log

elif experiment == 'ramanujan':
    # Search for continued fraction formulas at degree 3, range 5
    os.makedirs('scripts/experiments/ramanujan-machine/results', exist_ok=True)
    !./ramanujan_gpu 3 5 | tee scripts/experiments/ramanujan-machine/results/ramanujan_deg3_r5_colab.log

elif experiment == 'class_numbers':
    # Class numbers for a small range (first 10M discriminants)
    os.makedirs('data/class-numbers', exist_ok=True)
    !./class_v2 0 10000000 | tee data/class-numbers/class_colab.log


In [ ]:
# === Step 12: Harvest your results ===
# The agent picks up the log file and analyzes it
!python3 scripts/research_agent.py --once --phase harvest


In [ ]:
# === Step 13: Download your results to submit as a PR ===
# Your computation extended the frontier. Download the log files
# and submit them as a PR to cahlen/idontknow.

import glob
logs = glob.glob('scripts/experiments/*/results/*colab*.log') + glob.glob('data/**/colab*.log', recursive=True)
reviews = glob.glob('docs/verifications/*review*.json')

print(f'Your results: {len(logs)} experiment log(s), {len(reviews)} review(s)\n')
for f in logs + reviews[-5:]:
    print(f'  {f}')

# In Colab, use files.download() to get them
try:
    from google.colab import files
    print('\nClick to download:')
    for f in logs:
        files.download(f)
except:
    print('\nFiles are in the paths above. Download and submit a PR to:')
    print('  https://github.com/cahlen/idontknow')


In [ ]:
# === Step 10: Peer review a finding ===
# First, discover all available findings:
import glob, os, re

findings_dir = '../bigcompute.science/src/content/findings' if os.path.exists('../bigcompute.science') else None
slugs = []

# Try local findings directory first
if findings_dir:
    for f in sorted(glob.glob(f'{findings_dir}/*.md')):
        with open(f) as fh:
            for line in fh:
                if line.startswith('slug:'):
                    slugs.append(line.split(':', 1)[1].strip().strip('"'))
                    break

# Fallback: scan review files already in the repo
if not slugs:
    for f in glob.glob('docs/verifications/*_peer-review.json'):
        slug = os.path.basename(f).replace('_peer-review.json', '')
        if slug not in slugs:
            slugs.append(slug)

print(f'Found {len(slugs)} findings:\n')
for i, s in enumerate(slugs):
    print(f'  {i:2d}. {s}')

# Pick one to review (change the index or type a slug)
finding = slugs[0] if slugs else 'zaremba-density-phase-transition'
print(f'\nSelected: {finding}')
print('Change the line above to review a different finding')

# --- Run the review ---
if os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'):
    key = os.environ.get('GEMINI_API_KEY', os.environ.get('GOOGLE_API_KEY', ''))
    os.environ['API_KEY'] = key
    model, provider, api_base = 'gemini-2.5-flash', 'google', 'https://generativelanguage.googleapis.com/v1beta/openai'
elif os.environ.get('OPENAI_API_KEY'):
    os.environ['API_KEY'] = os.environ['OPENAI_API_KEY']
    model, provider, api_base = 'gpt-4.1', 'openai', 'https://api.openai.com/v1'
elif os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['API_KEY'] = os.environ['ANTHROPIC_API_KEY']
    model, provider, api_base = 'claude-sonnet-4-20250514', 'anthropic', 'https://api.anthropic.com/v1'
else:
    raise ValueError('No API key set')

print(f'\nReviewing {finding} with {model} ({provider})...')
!python3 scripts/reviews/run_review.py --slug {finding} --model {model} --provider {provider} --api-base {api_base} --skip-existing


In [ ]:
# === Step 11: View all reviews across all findings ===
import json, glob, os

# Collect all review files grouped by finding
reviews_by_slug = {}
for f in sorted(glob.glob('docs/verifications/*review*.json')):
    try:
        with open(f) as fh:
            r = json.load(fh)
        slug = r.get('finding_slug', '?')
        model = r.get('reviewer', {}).get('model', '?')
        verdict = r.get('overall_verdict', '?')
        cert = r.get('certification_recommendation', '?')
        if slug not in reviews_by_slug:
            reviews_by_slug[slug] = []
        reviews_by_slug[slug].append((model, verdict, cert))
    except:
        continue

print(f'{sum(len(v) for v in reviews_by_slug.values())} reviews across {len(reviews_by_slug)} findings:\n')
for slug in sorted(reviews_by_slug.keys()):
    revs = reviews_by_slug[slug]
    models = ', '.join(set(r[0] for r in revs if r[0] != '?'))
    worst = min(revs, key=lambda r: {'REJECT':0,'REVISE_AND_RESUBMIT':1,'ACCEPT_WITH_REVISION':2,'ACCEPT':3}.get(r[1], 2))
    print(f'  {slug[:45]:45s} {len(revs)} reviews  {worst[1]:25s} [{models}]')


In [ ]:
# === Step 12: Rebuild manifest (aggregate all reviews) ===
!python3 scripts/reviews/aggregate.py
!python3 scripts/reviews/validate.py --all

In [ ]:
# === Step 13: Review ALL findings at once ===
# This will take a few minutes (15 findings x ~30s each)
# Gemini free tier: 15 RPM, so this stays within limits

if os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'):
    key = os.environ.get('GEMINI_API_KEY', os.environ.get('GOOGLE_API_KEY', ''))
    os.environ['API_KEY'] = key
    !python3 scripts/reviews/run_review.py --all --model gemini-2.5-flash --provider google --api-base https://generativelanguage.googleapis.com/v1beta/openai --skip-existing
elif os.environ.get('OPENAI_API_KEY'):
    os.environ['API_KEY'] = os.environ['OPENAI_API_KEY']
    !python3 scripts/reviews/run_review.py --all --model gpt-4.1 --provider openai --skip-existing
else:
    print('Set GEMINI_API_KEY or OPENAI_API_KEY first')

## Contribute Your Reviews

**Your review adds a new perspective.** Different models catch different errors. Claude found 6 issues in Round 1. o3-pro found the prime count error. Your Gemini review might find something they all missed.

**To submit:**
1. Download the review JSON(s) from `docs/verifications/`
2. Open a PR to [cahlen/idontknow](https://github.com/cahlen/idontknow)
3. Your review will be added to the audit ledger on [bigcompute.science/verification](https://bigcompute.science/verification/)

**To run the full autonomous loop** (needs a machine with GPUs):
```bash
export GEMINI_API_KEY='AIza...'  # or OPENAI_API_KEY or ANTHROPIC_API_KEY
nohup ./scripts/run_agent.sh --loop 10m &
```

**More info:** [AGENTS.md](https://github.com/cahlen/idontknow/blob/main/AGENTS.md) · [Verification](https://bigcompute.science/verification/) · [MCP Server](https://mcp.bigcompute.science/mcp) (23 tools)